In [0]:
# 1. leemos el archivo csv
#display(dbutils.fs.ls("abfss://bronze@lsdata01.dfs.core.windows.net/"))
#abfss://bronze@lsdata01.dfs.core.windows.net/language.csv

#definimos los tipos de datos del archivo
from pyspark.sql.types import StringType, IntegerType, TimestampType
language_schema = StructType([
    StructField("languageId", IntegerType(), True),
    StructField("languageCode", StringType(), True),
    StructField("languageName", StringType(), True)
 ] )

#cargamos el df indicando los tipos de datos y formato del archivo como la cavecera
language_df = spark.read\
    .option("header", True)\
    .schema(language_schema)\
    .csv("abfss://bronze@lsdata01.dfs.core.windows.net/language.csv")

language_df.printSchema()

In [0]:
# Paso 2 - Seleccionar las columnas que se requieren

language_selected_df = language_df.select("languageId", "languageName")
language_selected_df.printSchema()


In [0]:
# Paso 3 - Renombrar Columnas

language_renamed_df = language_selected_df\
    .withColumnRenamed("languageId", "language_id")\
    .withColumnRenamed("languageName", "language_name")

display(language_renamed_df)

In [0]:
# Paso 4 - Añadir columnas a una tabla
from pyspark.sql.functions import current_timestamp, lit

language_final_df = language_renamed_df\
    .withColumn("ingestion_date", current_timestamp())\
    .withColumn("enviroment", lit("produccion"))

display(language_final_df)
language_final_df.printSchema()

In [0]:
# Paso 5 - Guardar datos en datalake en formato parket

language_final_df.write.mode("overwrite").format("parquet").save("abfss://silver@lsdata01.dfs.core.windows.net/language")

In [0]:
%fs
ls abfss://silver@lsdata01.dfs.core.windows.net/language

In [0]:
df = spark.read.parquet("abfss://silver@lsdata01.dfs.core.windows.net/language")
display(df)

In [0]:
%fs
ls abfss://silver@lsdata01.dfs.core.windows.net/language
